In [123]:
import pandas as pd 
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/amazonLabelled.csv")
df.head()

In [ ]:
df = df[['Feedback','Sentiment']]

In [126]:
df['Sentiment']=df['Sentiment'].apply(lambda x: 1 if x=='Positive' else 0)
df.head()

,Feedback,Sentiment
0,"Good case, Excellent value.",1
1,Great for the jawbone.,1
2,Tied to charger for conversations lasting more...,0
3,The mic is great.,1
4,I have to jiggle the plug to get it to line up...,0


In [127]:
X = df['Feedback']
y = df['Sentiment']

In [128]:
import tensorflow as tf

In [129]:
from tensorflow.keras.layers import Embedding,LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot

In [130]:
import nltk
import re
from nltk.corpus import stopwords

In [131]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [132]:
sample = X.copy()

In [133]:
'''
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(sample)):
    feedback = re.sub('[^a-zA-Z]', ' ', sample[i])
    feedback = feedback.lower()
    feedback = feedback.split()
    feedback = [ps.stem(word) for word in feedback if not word in stopwords.words('english')]
    feedback = ' '.join(feedback)
    corpus.append(feedback)
'''    

"\nfrom nltk.stem.porter import PorterStemmer\nps = PorterStemmer()\ncorpus = []\nfor i in range(0, len(sample)):\n    feedback = re.sub('[^a-zA-Z]', ' ', sample[i])\n    feedback = feedback.lower()\n    feedback = feedback.split()\n    feedback = [ps.stem(word) for word in feedback if not word in stopwords.words('english')]\n    feedback = ' '.join(feedback)\n    corpus.append(feedback)\n"

In [134]:
from nltk.stem import WordNetLemmatizer
wordnet = WordNetLemmatizer()
corpus = []
for i in range (len(sample)):
    feedback = re.sub("[^a-zA-Z]"," ",sample[i])
    feedback = feedback.lower()
    feedback = feedback.split()
    feedback = [wordnet.lemmatize(word) for word in feedback if word not in set(stopwords.words("english"))]
    feedback = " ".join(feedback)
    corpus.append(feedback)
   

In [135]:
voc_size = 5000

In [136]:
onehot_repr=[one_hot(words,voc_size)for words in corpus] 

In [137]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
embedded_docs

array([[   0,    0,    0, ..., 4805, 4680, 1667],
       [   0,    0,    0, ...,    0, 2110, 2767],
       [   0,    0,    0, ..., 1685, 1714, 1039],
       ...,
       [   0,    0,    0, ..., 4519, 3909, 4550],
       [   0,    0,    0, ..., 2332, 4638, 1643],
       [   0,    0,    0, ...,  726, 2093, 3204]], dtype=int32)

MOdel - Sequential

In [138]:
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 20, 40)            200000    
                                                                 
 lstm_5 (LSTM)               (None, 100)               56400     
                                                                 
 dense_5 (Dense)             (None, 1)                 101       
                                                                 
Total params: 256,501
Trainable params: 256,501
Non-trainable params: 0
_________________________________________________________________
None


In [139]:
len(embedded_docs),y.shape

(999, (999,))

In [140]:
X=np.array(embedded_docs)
y=np.array(y)

In [141]:
X.shape

(999, 20)

In [142]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [143]:
model.fit(X,y,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
16/16 [==============================] - 4s 76ms/step - loss: 0.6932 - accuracy: 0.4955 - val_loss: 0.6857 - val_accuracy: 0.5576
Epoch 2/10
16/16 [==============================] - 1s 41ms/step - loss: 0.6806 - accuracy: 0.7227 - val_loss: 0.6629 - val_accuracy: 0.8606
Epoch 3/10
16/16 [==============================] - 1s 39ms/step - loss: 0.6224 - accuracy: 0.8438 - val_loss: 0.5267 - val_accuracy: 0.7667
Epoch 4/10
16/16 [==============================] - 1s 40ms/step - loss: 0.4760 - accuracy: 0.8298 - val_loss: 0.3976 - val_accuracy: 0.8818
Epoch 5/10
16/16 [==============================] - 1s 38ms/step - loss: 0.3253 - accuracy: 0.9059 - val_loss: 0.2581 - val_accuracy: 0.9121
Epoch 6/10
16/16 [==============================] - 1s 39ms/step - loss: 0.2639 - accuracy: 0.9039 - val_loss: 0.2202 - val_accuracy: 0.9152
Epoch 7/10
16/16 [==============================] - 1s 37ms/step - loss: 0.1859 - accuracy: 0.9239 - val_loss: 0.1552 - val_accuracy: 0.9515
Epoch 8/10
16